# Phase 1 提取结果分析

分析5%探索阶段提取的知识点质量，检查：
1. Relation和Entity的分布
2. 是否过度依赖prompt例子
3. 知识点的多样性和泛化性
4. 具体样例检查

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
from pathlib import Path

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 设置样式
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. 加载数据

In [ ]:
# 加载提取结果
result_file = Path('../results/phase1_5percent_exploration.json')
with open(result_file, 'r') as f:
    data = json.load(f)

print(f"总电影数: {len(data['results'])}")
print(f"成功提取: {sum(1 for r in data['results'] if r['status'] == 'success')}")
print(f"提取失败: {sum(1 for r in data['results'] if r['status'] == 'error')}")

# 只看成功的结果
successful_results = [r for r in data['results'] if r['status'] == 'success']
print(f"\n成功提取的电影数: {len(successful_results)}")

## 2. 基本统计

In [ ]:
# 统计知识点数量
kp_counts = [r['num_knowledge_points'] for r in successful_results]

print(f"知识点统计:")
print(f"  总数: {sum(kp_counts)}")
print(f"  平均: {sum(kp_counts) / len(kp_counts):.2f}")
print(f"  最小: {min(kp_counts)}")
print(f"  最大: {max(kp_counts)}")
print(f"  中位数: {sorted(kp_counts)[len(kp_counts)//2]}")

# 知识点数量分布
plt.figure(figsize=(10, 5))
plt.hist(kp_counts, bins=range(min(kp_counts), max(kp_counts)+2), edgecolor='black', alpha=0.7)
plt.xlabel('Number of Knowledge Points per Movie')
plt.ylabel('Frequency')
plt.title('Distribution of Knowledge Points per Movie')
plt.axvline(sum(kp_counts)/len(kp_counts), color='red', linestyle='--', label=f'Mean: {sum(kp_counts)/len(kp_counts):.2f}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. Relation分析 - 关键问题！

In [ ]:
# Prompt中的例子relations
prompt_example_relations = [
    'color_scheme',
    'visual_style', 
    'main_element',
    'composition',
    'mood',
    'lighting',
    'typography',
    'texture'
]

# 收集所有relation
all_relations = []
for result in successful_results:
    for kp in result['knowledge_points']:
        all_relations.append(kp['relation'])

relation_counts = Counter(all_relations)

print(f"\n=== Relation分布分析 ===")
print(f"唯一relation数量: {len(relation_counts)}")
print(f"\nTop 20 最常见的relations:")
for rel, count in relation_counts.most_common(20):
    in_prompt = "✓ [Prompt例子]" if rel in prompt_example_relations else ""
    percentage = count / len(all_relations) * 100
    print(f"  {rel:25s}: {count:4d} ({percentage:5.2f}%) {in_prompt}")

# 计算有多少是prompt例子中的
prompt_relation_count = sum(relation_counts[rel] for rel in prompt_example_relations if rel in relation_counts)
total_relations = len(all_relations)
prompt_percentage = prompt_relation_count / total_relations * 100

print(f"\n⚠️ 关键指标:")
print(f"  Prompt例子中的8个relation占比: {prompt_percentage:.2f}%")
print(f"  非Prompt例子的relation占比: {100-prompt_percentage:.2f}%")
print(f"  非Prompt例子的唯一relation数: {len([r for r in relation_counts if r not in prompt_example_relations])}")

In [ ]:
# 可视化Top 15 relations
top_relations = relation_counts.most_common(15)
relations, counts = zip(*top_relations)

# 标记是否在prompt例子中
colors = ['#ff6b6b' if rel in prompt_example_relations else '#4ecdc4' for rel in relations]

plt.figure(figsize=(12, 6))
bars = plt.barh(range(len(relations)), counts, color=colors)
plt.yticks(range(len(relations)), relations)
plt.xlabel('Count')
plt.title('Top 15 Relations (Red = In Prompt Examples, Blue = Not in Prompt)')
plt.gca().invert_yaxis()

# 添加图例
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#ff6b6b', label='In Prompt Examples'),
    Patch(facecolor='#4ecdc4', label='Not in Prompt')
]
plt.legend(handles=legend_elements, loc='lower right')
plt.tight_layout()
plt.show()

## 4. Entity分析

In [ ]:
# 收集所有entity
all_entities = []
relation_entity_map = defaultdict(list)

for result in successful_results:
    for kp in result['knowledge_points']:
        entity = kp['entity']
        relation = kp['relation']
        all_entities.append(entity)
        relation_entity_map[relation].append(entity)

entity_counts = Counter(all_entities)

print(f"\n=== Entity分布分析 ===")
print(f"唯一entity数量: {len(entity_counts)}")
print(f"总entity数量: {len(all_entities)}")
print(f"平均每个entity出现次数: {len(all_entities) / len(entity_counts):.2f}")

print(f"\nTop 30 最常见的entities:")
for entity, count in entity_counts.most_common(30):
    percentage = count / len(all_entities) * 100
    print(f"  {entity:35s}: {count:4d} ({percentage:5.2f}%)")

In [ ]:
# 检查每个主要relation的entity多样性
print(f"\n=== 主要Relation的Entity多样性 ===")
for relation in relation_counts.most_common(10):
    rel_name = relation[0]
    entities = relation_entity_map[rel_name]
    unique_entities = len(set(entities))
    total_entities = len(entities)
    
    print(f"\n{rel_name}:")
    print(f"  总数: {total_entities}, 唯一: {unique_entities}, 多样性: {unique_entities/total_entities:.2%}")
    
    # Top 5 entities for this relation
    entity_counter = Counter(entities)
    print(f"  Top 5 entities:")
    for ent, cnt in entity_counter.most_common(5):
        print(f"    - {ent}: {cnt}")

## 5. 随机样例检查

In [ ]:
# 随机抽取5个电影查看提取结果
import random
random.seed(42)

sample_results = random.sample(successful_results, min(5, len(successful_results)))

print("\n=== 随机样例检查 ===")
for i, result in enumerate(sample_results, 1):
    print(f"\n{'='*60}")
    print(f"样例 {i}: RecBole ID {result['recbole_id']} (Movie {result['original_movie_id']})")
    print(f"知识点数量: {result['num_knowledge_points']}")
    print(f"\n知识点列表:")
    for kp in result['knowledge_points']:
        in_prompt = "[P]" if kp['relation'] in prompt_example_relations else "   "
        print(f"  {in_prompt} {kp['relation']:20s}: {kp['entity']}")
    
    print(f"\n原始输出:")
    print(result['raw_output'][:300] + "..." if len(result['raw_output']) > 300 else result['raw_output'])

## 6. 质量问题诊断

In [ ]:
print("\n" + "="*60)
print("质量诊断报告")
print("="*60)

# 1. Prompt依赖度
prompt_dependency = prompt_percentage
print(f"\n1. Prompt依赖度: {prompt_dependency:.1f}%")
if prompt_dependency > 80:
    print("   ⚠️ 警告: 严重依赖prompt例子，缺乏多样性")
elif prompt_dependency > 60:
    print("   ⚠️ 注意: 较多依赖prompt例子")
else:
    print("   ✓ 良好: prompt依赖度适中")

# 2. Relation多样性
unique_relation_count = len(relation_counts)
print(f"\n2. Relation多样性: {unique_relation_count} 个唯一relation")
if unique_relation_count < 15:
    print("   ⚠️ 警告: relation类型过少，缺乏多样性")
elif unique_relation_count < 30:
    print("   ⚠️ 注意: relation类型偏少")
else:
    print("   ✓ 良好: relation类型丰富")

# 3. Entity多样性
unique_entity_count = len(entity_counts)
avg_entity_reuse = len(all_entities) / unique_entity_count
print(f"\n3. Entity多样性: {unique_entity_count} 个唯一entity")
print(f"   平均每个entity被使用 {avg_entity_reuse:.2f} 次")
if avg_entity_reuse > 3:
    print("   ⚠️ 警告: entity重复使用率高，可能缺乏特异性")
elif avg_entity_reuse > 2:
    print("   ⚠️ 注意: entity重复使用率偏高")
else:
    print("   ✓ 良好: entity多样性好")

# 4. 知识点数量
avg_kps = sum(kp_counts) / len(kp_counts)
print(f"\n4. 平均知识点数量: {avg_kps:.1f}")
if avg_kps < 10:
    print("   ⚠️ 警告: 知识点数量偏少")
elif avg_kps > 15:
    print("   ⚠️ 注意: 知识点数量偏多，可能有冗余")
else:
    print("   ✓ 良好: 知识点数量适中")

print("\n" + "="*60)
print("\n建议:")
if prompt_dependency > 70:
    print("1. ⚠️ 修改prompt，减少或删除具体的relation例子")
    print("2. 让模型更自由地发现relation类型")
if unique_relation_count < 20:
    print("3. 增加prompt的开放性，鼓励提取更多样化的relation")
if avg_entity_reuse > 2.5:
    print("4. 检查entity是否足够具体和有区分度")

## 7. 保存分析报告

In [ ]:
# 生成分析报告
report = {
    'total_movies': len(successful_results),
    'total_knowledge_points': sum(kp_counts),
    'avg_kps_per_movie': sum(kp_counts) / len(kp_counts),
    'unique_relations': len(relation_counts),
    'unique_entities': len(entity_counts),
    'prompt_dependency_percentage': prompt_percentage,
    'avg_entity_reuse': avg_entity_reuse,
    'top_20_relations': relation_counts.most_common(20),
    'top_30_entities': entity_counts.most_common(30)
}

report_file = Path('../results/phase1_analysis_report.json')
with open(report_file, 'w') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print(f"\n分析报告已保存到: {report_file}")